In [ ]:
import polars as pl
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from optbinning import BinningProcess
import lightgbm as lgb
import mlflow
import warnings
warnings.filterwarnings('ignore')

mlflow.set_tracking_uri('mlruns')
mlflow.set_experiment('task1')
RUN_TAG = 'mar23'

In [ ]:
it = pl.read_csv('task1/data/IT.csv')
oot = pl.read_csv('task1/data/OOT.csv')

num_cols = [c for c in it.columns if c.startswith('NUMERICAL_')]
it = it.with_columns(pl.col('TIME').str.to_datetime().alias('TIME_DT'))
oot = oot.with_columns(pl.col('TIME').str.to_datetime().alias('TIME_DT'))

cutoff = it.sort('TIME_DT')['TIME_DT'].quantile(0.8)
train = it.filter(pl.col('TIME_DT') <= cutoff)
val = it.filter(pl.col('TIME_DT') > cutoff)
print(f'Train: {train.shape[0]}, Val: {val.shape[0]}')

X_train = train.select(num_cols).to_pandas()
y_train = train['TARGET'].to_numpy()
X_val = val.select(num_cols).to_pandas()
y_val = val['TARGET'].to_numpy()
X_oot = oot.select(num_cols).to_pandas()

In [ ]:
bp = BinningProcess(
    variable_names=num_cols,
    min_prebin_size=0.02,
    min_bin_size=0.05,
    max_n_bins=10,
    selection_criteria={'iv': {'min': 0.02}}
)
bp.fit(X_train.values, y_train)
selected = list(bp.get_support(names=True))
print(f'Selected (IV>=0.02): {len(selected)} -> {selected}')

summary = bp.summary().sort_values('iv', ascending=False)
print(summary[['name', 'iv']].head(10).to_string())

In [ ]:
X_train_woe = bp.transform(X_train.values, metric='woe')
X_val_woe = bp.transform(X_val.values, metric='woe')
X_oot_woe = bp.transform(X_oot.values, metric='woe')

for C in [0.01, 0.1, 1.0, 10.0]:
    lr = LogisticRegression(C=C, max_iter=1000, solver='lbfgs')
    lr.fit(X_train_woe, y_train)
    vg = 2 * roc_auc_score(y_val, lr.predict_proba(X_val_woe)[:, 1]) - 1
    print(f'WOE+LR C={C}: val_gini={vg:.4f}')

In [ ]:
lgb_train = lgb.Dataset(X_train[num_cols], y_train)
lgb_val = lgb.Dataset(X_val[num_cols], y_val, reference=lgb_train)

params = {
    'objective': 'binary',
    'metric': 'auc',
    'verbosity': -1,
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'min_child_samples': 100,
}

model = lgb.train(
    params, lgb_train, num_boost_round=500,
    valid_sets=[lgb_val], callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)]
)

p_val_lgb = model.predict(X_val[num_cols])
val_gini_lgb = 2 * roc_auc_score(y_val, p_val_lgb) - 1
print(f'LightGBM raw: val_gini={val_gini_lgb:.4f}')

In [ ]:
lgb_train_woe = lgb.Dataset(X_train_woe, y_train)
lgb_val_woe = lgb.Dataset(X_val_woe, y_val, reference=lgb_train_woe)

params_woe = {
    'objective': 'binary',
    'metric': 'auc',
    'verbosity': -1,
    'num_leaves': 15,
    'learning_rate': 0.05,
    'min_child_samples': 100,
}

model_woe = lgb.train(
    params_woe, lgb_train_woe, num_boost_round=300,
    valid_sets=[lgb_val_woe], callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)]
)

p_val_lgb_woe = model_woe.predict(X_val_woe)
val_gini_lgb_woe = 2 * roc_auc_score(y_val, p_val_lgb_woe) - 1
print(f'LightGBM WOE: val_gini={val_gini_lgb_woe:.4f}')

In [ ]:
lr_best = LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs')
lr_best.fit(X_train_woe, y_train)
p_lr = lr_best.predict_proba(X_val_woe)[:, 1]

p_ensemble = 0.5 * p_lr + 0.5 * p_val_lgb
val_gini_ens = 2 * roc_auc_score(y_val, p_ensemble) - 1
print(f'Ensemble (LR+LGB): val_gini={val_gini_ens:.4f}')

results = {
    'WOE+LR': 2 * roc_auc_score(y_val, p_lr) - 1,
    'LGB_raw': val_gini_lgb,
    'LGB_woe': val_gini_lgb_woe,
    'Ensemble': val_gini_ens
}
best_name = max(results, key=results.get)
best_gini = results[best_name]
print(f'\nBest: {best_name} = {best_gini:.4f}')
for k, v in sorted(results.items(), key=lambda x: -x[1]):
    print(f'  {k}: {v:.4f}')

In [ ]:
if best_name == 'WOE+LR':
    p_oot_final = lr_best.predict_proba(X_oot_woe)[:, 1]
elif best_name == 'LGB_raw':
    p_oot_final = model.predict(X_oot[num_cols])
elif best_name == 'LGB_woe':
    p_oot_final = model_woe.predict(X_oot_woe)
else:
    p_oot_final = 0.5 * lr_best.predict_proba(X_oot_woe)[:, 1] + 0.5 * model.predict(X_oot[num_cols])

val_gini = best_gini

preds = pl.DataFrame({'ID_APPLICATION': oot['ID_APPLICATION'], 'SCORE': p_oot_final})
preds.write_csv('task1/predictions.csv')
print(f'OOT predictions saved: {preds.shape}')

In [ ]:
with mlflow.start_run(run_name='v2_compare_models'):
    mlflow.log_param('model', best_name)
    mlflow.log_param('n_features_woe', len(selected))
    mlflow.log_param('n_features_raw', len(num_cols))
    mlflow.log_metric('val_gini', val_gini)
    for k, v in results.items():
        mlflow.log_metric(f'val_gini_{k}', v)
    mlflow.set_tag('task', 'task1')
    mlflow.set_tag('run_tag', RUN_TAG)
print(f'val_gini: {val_gini:.4f}')